# FineTunePilot — Stage 2: evaluate base vs tuned on Kaggle

Scores `Qwen/Qwen2.5-0.5B-Instruct` on the 20-row test set, before and after the QLoRA adapter,
then prints a comparison. Metrics: `json_valid`, `tool_name_accuracy`, `args_key_match`,
`args_exact_match`, `format_ok` (all 0..1).

**Before you start:** Accelerator → **GPU T4**, **Internet on**.
Kaggle sessions start empty, so if no adapter exists yet, step 4 trains one (a few minutes).

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the code and install it
Skips the clone if the repo is already here from notebook 01 in this session.

In [ ]:
import os
%cd /kaggle/working
if not os.path.isdir("Finetune-pilot-ai"):
    !git clone --depth 1 --branch claude/zen-meitner-hetiul https://github.com/manojastro/Finetune-pilot-ai.git
%cd Finetune-pilot-ai
!pip install -q -e ".[gpu]"

## 3. One GPU only, and library versions
Needs transformers 5.x, trl 1.x, and `cuda True`.

In [ ]:
%env CUDA_VISIBLE_DEVICES=0
!python -c "import torch, transformers, trl, peft; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| transformers', transformers.__version__, 'trl', trl.__version__, 'peft', peft.__version__)"

## 4. Find the trained adapter (train one if missing)
Uses the newest successful real training run in `runs/`.

In [ ]:
import glob, json

def latest_adapter():
    for path in sorted(glob.glob("runs/*.json"), reverse=True):
        log = json.load(open(path))
        if log["kind"] == "train" and log["status"] == "success" and not log["dry_run"]:
            return log["artifacts"]["adapter_dir"], path
    return None, None

adapter, train_run = latest_adapter()
if adapter is None:
    !finetunepilot train --config configs/qwen05b_toolcall.yaml
    adapter, train_run = latest_adapter()
print("adapter:", adapter, "| train run:", train_run)

## 5. Evaluate the base model
Same 4-bit loading as training, greedy decoding, `max_new_tokens=128` (the longest target is 58 tokens).

In [ ]:
!finetunepilot eval --base-model Qwen/Qwen2.5-0.5B-Instruct --test data/sample/test.jsonl --name eval-base

## 6. Evaluate the tuned model (base + adapter)

In [ ]:
!finetunepilot eval --base-model Qwen/Qwen2.5-0.5B-Instruct --adapter {adapter} --test data/sample/test.jsonl --name eval-tuned

## 7. Compare base vs tuned

In [ ]:
def latest(prefix):
    paths = sorted(p for p in glob.glob(f"runs/{prefix}-*.json") if "-dryrun-" not in p)
    return paths[-1]

base_run, tuned_run = latest("eval-base"), latest("eval-tuned")
!COLUMNS=160 finetunepilot compare {base_run} {tuned_run}

## 8. Copy this back into Claude Code
Metrics for both runs, the training summary, and every wrong prediction (not the full JSON, which is long).

In [ ]:
def brief(path):
    log = json.load(open(path))
    out = {k: log[k] for k in ("run_id", "kind", "status", "duration_s", "metrics", "errors", "library_versions", "hardware")}
    if log["kind"] == "eval":
        out["config"] = log["config"]
        out["wrong"] = [{k: p[k] for k in ("id", "request", "target", "output")} for p in log["predictions"] if not p["args_exact_match"]]
    return out

print(json.dumps({"train": brief(train_run), "base": brief(base_run), "tuned": brief(tuned_run)}, indent=1))